# AMR Cascade Platform Debug Notebook

Purpose: inspect the current pipeline inputs, intermediate artifacts, model inputs, and final outputs before running production on HPC.

This notebook is intentionally audit-oriented. It should answer four questions:

1. What data layers exist right now?
2. Which pairs enter cascade estimation and validation?
3. Does adjusted downstream-testing regression use only validated `robust` / `supported` pairs?
4. Does prevalence-shift analysis run only for downstream antibiotics from validated escalation edges?

Default behavior is read-only. It does not regenerate artifacts unless you explicitly set `RUN_OPTIONAL_STAGES = True`.


## 0. Setup

Use the same config/context loader as the CLI. If this fails, the project environment is not ready and HPC should not be started.


In [1]:
import sys
from pathlib import Path
import json
import math

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "amr_cascade_platform").exists():
    PROJECT_ROOT = Path("/Users/awotoroebenezer/Desktop/amr_cascade_platform")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from amr_cascade_platform.cli.main import build_context
from amr_cascade_platform.core.utils.antibiotic_names import normalize_antibiotic_label

settings, path_manager, mapper = build_context(PROJECT_ROOT, "mac")

ORGANISM_LABEL = "ESCHERICHIA COLI"
ORGANISM_SLUG = "escherichia_coli"
SCOPE = "combined"
SITE = None
RUN_OPTIONAL_STAGES = False

print("project:", PROJECT_ROOT)
print("environment:", settings.environment.name)
print("data root:", path_manager.paths.raw.parent)
print("organism label:", ORGANISM_LABEL)
print("organism slug:", ORGANISM_SLUG)


project: /Users/awotoroebenezer/Desktop/amr_cascade_platform
environment: mac
data root: /Users/awotoroebenezer/Desktop/amr_cascade_platform/data
organism label: ESCHERICHIA COLI
organism slug: escherichia_coli


## 1. Current Scientific Contract

The corrected production contract is:

- **Cascade estimation** starts with episode-pair rows.
- **Retention** uses raw observation evidence only: support, informative downstream testing, panel-bundling status, and raw ER.
- **Validation** assigns `robust`, `supported`, `mixed`, or `insufficient`.
- **Adjusted downstream-testing regression** is fitted **after validation** and only for `robust` / `supported` pairs.
- **Prevalence-shift analysis** is restricted to downstream antibiotics from validated **escalation** edges.
- Suppression edges can be valid cascade findings, but they are not used as resistance-triggered prevalence anchors.


In [2]:
from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisWorkflow

print("Adjusted model helper:", CascadeAnalysisWorkflow._validated_edge_results.__name__)
print(CascadeAnalysisWorkflow._validated_edge_results.__doc__)


Adjusted model helper: _validated_edge_results
Restrict adjusted modeling to robust/supported validated pairs.

        DownstreamTestingRegression uses the rows it receives as its model
        universe. Passing all support-screened edges here would make RQ2 partly
        exploratory. Passing only robust/supported validation results makes RQ2
        exactly: among validated cascade edges, do adjusted associations remain
        directionally coherent?
        


## 2. Artifact Map

This cell lists the specific files the notebook expects. Missing files are not automatically an error; they mean that stage has not been run yet or was deleted before a fresh run.


In [3]:
def exists_table(path: Path) -> dict:
    return {"path": str(path), "exists": path.exists(), "size_mb": round(path.stat().st_size / 1e6, 2) if path.exists() else None}

gold_dir = path_manager.paths.gold / SCOPE / "organisms" / ORGANISM_SLUG
cascade_dir = path_manager.paths.artifacts / settings.cascade.outputs.result_dir / SCOPE / "organisms" / ORGANISM_SLUG
prevalence_dir = path_manager.paths.artifacts / settings.prevalence.output_dir / SCOPE / "organisms" / ORGANISM_SLUG
report_dir = path_manager.paths.outputs / "reports" / SCOPE / "organisms" / ORGANISM_SLUG

expected = {
    "gold_culture_episodes": gold_dir / "culture_episodes.parquet",
    "gold_culture_drug_episodes": gold_dir / "culture_drug_episodes.parquet",
    "gold_eligible_pairs": gold_dir / "eligible_pairs.parquet",
    "gold_drug_pair_episodes": gold_dir / "drug_pair_episodes.parquet",
    "cascade_edge_report": cascade_dir / "edge_report.parquet",
    "cascade_validation_results": cascade_dir / "validation_results.parquet",
    "cascade_adjusted_diagnostics": cascade_dir / "adjusted_model_diagnostics.parquet",
    "prevalence_shift": prevalence_dir / "prevalence_shift.parquet",
        "prevalence_legacy_delta_curve": prevalence_dir / "legacy" / "legacy_delta_sensitivity_curves.parquet",
    "prevalence_mnar_curve": prevalence_dir / "prevalence_mnar_sensitivity_curves.parquet",
    "report_manifest": report_dir / "report_manifest.json",
}

pd.DataFrame([{"artifact": k, **exists_table(v)} for k, v in expected.items()])


,artifact,path,exists,size_mb
0,gold_culture_episodes,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,2.87
1,gold_culture_drug_episodes,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,3.24
2,gold_eligible_pairs,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,4.12
3,gold_drug_pair_episodes,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,4.88
4,cascade_edge_report,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.28
5,cascade_validation_results,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.12
6,cascade_adjusted_diagnostics,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.07
7,prevalence_shift,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.05
8,prevalence_legacy_delta_curve,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.04
9,prevalence_mnar_curve,/Users/awotoroebenezer/Desktop/amr_cascade_pla...,True,0.03


## 3. Gold Layer Inspection

This verifies the denominator objects before any model is considered:

- `culture_episodes`: organism-level analysis units.
- `culture_drug_episodes`: directly observed AST rows.
- `eligible_pairs`: episode-drug opportunity denominator.
- `drug_pair_episodes`: directional upstream-downstream pair rows.


In [4]:
def read_parquet_if_exists(path: Path, columns=None) -> pd.DataFrame:
    if not path.exists():
        print("MISSING:", path)
        return pd.DataFrame()
    return pd.read_parquet(path, columns=columns)

culture_episodes = read_parquet_if_exists(expected["gold_culture_episodes"])
culture_drug_episodes = read_parquet_if_exists(expected["gold_culture_drug_episodes"])
eligible_pairs = read_parquet_if_exists(expected["gold_eligible_pairs"])
drug_pairs = read_parquet_if_exists(expected["gold_drug_pair_episodes"])

rows = []
for name, df in {
    "culture_episodes": culture_episodes,
    "culture_drug_episodes": culture_drug_episodes,
    "eligible_pairs": eligible_pairs,
    "drug_pair_episodes": drug_pairs,
}.items():
    rows.append({"table": name, "rows": len(df), "columns": len(df.columns), "empty": df.empty})

pd.DataFrame(rows)


,table,rows,columns,empty
0,culture_episodes,74889,9,False
1,culture_drug_episodes,104389,10,False
2,eligible_pairs,3070449,16,False
3,drug_pair_episodes,4175160,13,False


In [5]:
# Show schemas compactly.
for name, df in {
    "culture_episodes": culture_episodes,
    "culture_drug_episodes": culture_drug_episodes,
    "eligible_pairs": eligible_pairs,
    "drug_pair_episodes": drug_pairs,
}.items():
    print("\n==", name, "==")
    if df.empty:
        print("empty or missing")
    else:
        print(list(df.columns))
        display(df.head(3))



== culture_episodes ==
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered', 'organism', 'source_site', 'ordering_mode', 'culture_description', 'was_positive']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered,organism,source_site,ordering_mode,culture_description,was_positive
0,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,Outpatient,URINE,1
1,JC739701,131247304008,552834814,2018-03-03 06:12:00+00:00,ESCHERICHIA COLI,armd,Outpatient,URINE,1
2,JC650342,131024511713,419486271,2013-05-25 06:39:00+00:00,ESCHERICHIA COLI,armd,Inpatient,URINE,1



== culture_drug_episodes ==
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered', 'organism', 'source_site', 'antibiotic', 'susceptibility', 'was_tested', 'observation_layer']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered,organism,source_site,antibiotic,susceptibility,was_tested,observation_layer
0,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,MEROPENEM,SUSCEPTIBLE,1,observed_ast
1,JC739701,131247304008,552834814,2018-03-03 06:12:00+00:00,ESCHERICHIA COLI,armd,AMPICILLIN,RESISTANT,1,observed_ast
2,JC650342,131024511713,419486271,2013-05-25 06:39:00+00:00,ESCHERICHIA COLI,armd,AMPICILLIN,RESISTANT,1,observed_ast



== eligible_pairs ==
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered', 'organism', 'source_site', 'availability_era', 'antibiotic', 'is_observed_tested', 'is_intrinsic_resistance', 'is_biologically_eligible', 'availability_support_n', 'is_operationally_available', 'eligibility_denominator', 'is_eligible', 'observation_status']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered,organism,source_site,availability_era,antibiotic,is_observed_tested,is_intrinsic_resistance,is_biologically_eligible,availability_support_n,is_operationally_available,eligibility_denominator,is_eligible,observation_status
0,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,2015-2019,AMIKACIN,0,0,1,732,1,operational_site_organism_era,1,not_observed
1,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,2015-2019,AMOXICILLIN/CLAVULANIC ACID,0,0,1,818,1,operational_site_organism_era,1,not_observed
2,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,2015-2019,AMPICILLIN,0,0,1,909,1,operational_site_organism_era,1,not_observed



== drug_pair_episodes ==
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered', 'organism', 'source_site', 'upstream_antibiotic', 'upstream_susceptibility', 'downstream_antibiotic', 'downstream_tested', 'downstream_eligible', 'downstream_intrinsic_resistance', 'pair_direction']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered,organism,source_site,upstream_antibiotic,upstream_susceptibility,downstream_antibiotic,downstream_tested,downstream_eligible,downstream_intrinsic_resistance,pair_direction
0,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,MEROPENEM,SUSCEPTIBLE,AMIKACIN,0,1,0,MEROPENEM -> AMIKACIN
1,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,MEROPENEM,SUSCEPTIBLE,AMOXICILLIN/CLAVULANIC ACID,0,1,0,MEROPENEM -> AMOXICILLIN/CLAVULANIC ACID
2,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,MEROPENEM,SUSCEPTIBLE,AMPICILLIN,0,1,0,MEROPENEM -> AMPICILLIN


## 4. Denominator and Observation-State Audit

This checks whether the denominator is behaving as intended:

- `is_eligible == 1` defines the eligible opportunity space.
- Observed AST rows should be a subset of eligible opportunities after matching on the episode key plus antibiotic.
- Non-binary observed results are not binary-evaluable prevalence outcomes.


In [6]:
join_keys = list(settings.gold.episode_key_columns)
pair_keys = join_keys + ["antibiotic"]

if not eligible_pairs.empty:
    denom_summary = eligible_pairs.assign(
        antibiotic_norm=eligible_pairs["antibiotic"].map(normalize_antibiotic_label)
    ).groupby("is_eligible", dropna=False).size().reset_index(name="rows")
    display(denom_summary)

if not culture_drug_episodes.empty:
    observed_summary = (
        culture_drug_episodes.assign(antibiotic_norm=culture_drug_episodes["antibiotic"].map(normalize_antibiotic_label))
        .groupby("susceptibility", dropna=False)
        .size()
        .reset_index(name="observed_rows")
        .sort_values("observed_rows", ascending=False)
    )
    display(observed_summary.head(20))


,is_eligible,rows
0,0,750443
1,1,2320006


,susceptibility,observed_rows
3,SUSCEPTIBLE,86111
2,RESISTANT,12983
1,INTERMEDIATE,2667
4,<NA>,2600
0,INCONCLUSIVE,28


## 5. Episode-Pair Unit-of-Analysis Check

The pair table must have one row per configured episode/upstream/downstream pair. Duplicate rows inflate support, branch probabilities, validation, and prevalence triggers.


In [7]:
if not drug_pairs.empty:
    pair_unit_keys = join_keys + ["upstream_antibiotic", "downstream_antibiotic"]
    missing = [c for c in pair_unit_keys if c not in drug_pairs.columns]
    if missing:
        print("Cannot check uniqueness; missing columns:", missing)
    else:
        duplicate_n = int(drug_pairs.duplicated(pair_unit_keys).sum())
        print("duplicate episode/upstream/downstream rows:", duplicate_n)
        if duplicate_n:
            display(drug_pairs.loc[drug_pairs.duplicated(pair_unit_keys, keep=False), pair_unit_keys + ["upstream_susceptibility", "downstream_tested"]].head(20))
        else:
            print("OK: one row per episode/upstream/downstream pair.")


duplicate episode/upstream/downstream rows: 0
OK: one row per episode/upstream/downstream pair.


## 6. Cascade Estimation: Pre-Validation Inputs

This reproduces the cascade estimation steps without running the slow validation loop:

1. Panel-bundling / co-testing filter.
2. Conditional probabilities.
3. Escalation ratio.
4. Raw retention before validation.

Adjusted regression is deliberately **not** run here.


In [8]:
from amr_cascade_platform.cascade.analyzers.cotesting_filter_analyzer import CoTestingFilterAnalyzer
from amr_cascade_platform.cascade.analyzers.conditional_probability_analyzer import ConditionalProbabilityAnalyzer
from amr_cascade_platform.cascade.analyzers.escalation_ratio_analyzer import EscalationRatioAnalyzer
from amr_cascade_platform.cascade.analyzers.retained_edge_analyzer import RetainedEdgeAnalyzer
from amr_cascade_platform.cascade.statistics.downstream_testing_regression import DownstreamTestingRegression

if not drug_pairs.empty:
    filtered_pairs, cotesting_pairs = CoTestingFilterAnalyzer(settings).filter(drug_pairs)
    conditional_probabilities = ConditionalProbabilityAnalyzer(settings).analyze(filtered_pairs)
    escalation_results = EscalationRatioAnalyzer(settings).analyze(conditional_probabilities)
    empty_adjusted = DownstreamTestingRegression(settings, path_manager)._empty_results()
    retained_edges_pre_validation = RetainedEdgeAnalyzer(settings).analyze(escalation_results, empty_adjusted)

    print("drug_pair_episodes:", len(drug_pairs))
    print("filtered_pairs:", len(filtered_pairs))
    print("co-testing excluded rows/pairs table:", len(cotesting_pairs))
    print("conditional_probability rows:", len(conditional_probabilities))
    print("escalation_result rows:", len(escalation_results))
    print("retained pre-validation edges:", len(retained_edges_pre_validation))
    display(escalation_results.sort_values("escalation_ratio", ascending=False).head(10))
else:
    filtered_pairs = conditional_probabilities = escalation_results = retained_edges_pre_validation = pd.DataFrame()
    print("Gold pair table missing; run gold build first.")


drug_pair_episodes: 4175160
filtered_pairs: 4175160
co-testing excluded rows/pairs table: 0
conditional_probability rows: 2359
escalation_result rows: 954
retained pre-validation edges: 714


,upstream_antibiotic,downstream_antibiotic,upstream_result_group_x,positive_support_n,positive_tested_n,positive_probability,upstream_result_group_y,negative_support_n,negative_tested_n,negative_probability,total_support_n,escalation_ratio,passes_support_threshold
646,ERTAPENEM,PIPERACILLIN,positive,1,0,0.0,negative,1979,0,0.000000,1980,990.000000,False
625,ERTAPENEM,CEFOTETAN,positive,2,0,0.0,negative,2339,0,0.000000,2341,780.000000,False
4,AMIKACIN,CEFALOTIN,positive,1,0,0.0,negative,1451,0,0.000000,1452,726.000000,False
25,AMIKACIN,TICARCILLIN/CLAVULANIC ACID,positive,1,0,0.0,negative,1152,0,0.000000,1153,576.500000,False
22,AMIKACIN,PIPERACILLIN,positive,1,0,0.0,negative,973,0,0.000000,974,487.000000,False
16,AMIKACIN,FOSFOMYCIN,positive,1,0,0.0,negative,1927,1,0.000519,1928,321.333333,False
436,CEFTAZIDIM/AVIBACTAM,CEFALOTIN,positive,1,0,0.0,negative,614,0,0.000000,615,307.500000,False
636,ERTAPENEM,ERAVACYCLINE,positive,2,0,0.0,negative,2299,1,0.000435,2301,255.555556,False
637,ERTAPENEM,FOSFOMYCIN,positive,1,0,0.0,negative,2878,3,0.001042,2879,205.642857,False
836,NITROFURANTOIN,TICARCILLIN/CLAVULANIC ACID,positive,20,0,0.0,negative,2256,0,0.000000,2276,107.476190,True


## 7. Validation Results

Validation is the slow stage. This notebook reads the validation artifact if it exists. It does not rerun permutation/bootstrap by default.


In [9]:
validation_results = read_parquet_if_exists(expected["cascade_validation_results"])
edge_report = read_parquet_if_exists(expected["cascade_edge_report"])

if not validation_results.empty:
    display(validation_results["validation_status"].value_counts(dropna=False).rename_axis("validation_status").reset_index(name="edge_n"))
    display(validation_results.head())
else:
    print("No validation_results.parquet found. Adjusted-model and prevalence checks below will be limited.")

if not edge_report.empty:
    display(edge_report["validation_status"].value_counts(dropna=False).rename_axis("validation_status").reset_index(name="edge_n"))
    cols = [c for c in ["upstream_antibiotic", "downstream_antibiotic", "cascade_direction", "escalation_ratio", "validation_status", "adjusted_odds_ratio", "supports_adjusted_model"] if c in edge_report.columns]
    display(edge_report.loc[:, cols].head(20))


,validation_status,edge_n
0,mixed,492
1,insufficient,222


,upstream_antibiotic,downstream_antibiotic,cascade_direction,observed_escalation_ratio,permutation_p_value,permutation_fdr_q_value,permutation_fdr_supported,permutation_null_median_er,permutation_null_q95_er,bootstrap_median_er,...,between_site_permutation_p_value,between_site_permutation_fdr_q_value,between_site_permutation_supported,temporal_early_er,temporal_late_er,temporal_n_early,temporal_n_late,temporal_direction_agreement,temporal_log_ratio_delta,validation_status
0,AMPICILLIN,AMIKACIN,escalation,1.349429,0.039216,0.560000,False,0.947096,1.198684,1.338443,...,0.058824,0.184920,False,7.026616,1.214584,2106,2106,True,1.755304,mixed
1,AMPICILLIN,AMOXICILLIN/CLAVULANIC ACID,suppression,0.759446,0.019608,0.560000,False,0.948243,1.236120,0.788397,...,0.960784,1.000000,False,0.461206,0.777479,2106,2106,True,0.522211,mixed
2,AMPICILLIN,AMPICILLIN/SULBACTAM,escalation,1.665833,0.019608,0.560000,False,1.051083,1.246419,1.650100,...,0.784314,1.000000,False,3.011407,1.223675,2106,2106,True,0.900549,mixed
3,AMPICILLIN,AZTREONAM,suppression,0.960221,0.313725,0.631579,False,1.099815,1.473417,0.955866,...,0.019608,0.092974,False,11.041825,0.919485,2106,2106,False,2.485632,insufficient
4,AMPICILLIN,CEFALOTIN,escalation,1.036383,0.705882,0.756757,False,1.036383,6.218299,1.034754,...,0.039216,0.142126,False,0.873156,1.130872,1268,1268,False,0.258630,insufficient


,validation_status,edge_n
0,mixed,492
1,insufficient,222


,upstream_antibiotic,downstream_antibiotic,cascade_direction,escalation_ratio,validation_status,adjusted_odds_ratio,supports_adjusted_model
0,CEFTRIAXON,FOSFOMYCIN,escalation,70.525597,mixed,NaN,None
1,AZTREONAM,FOSFOMYCIN,escalation,53.960526,mixed,NaN,None
2,PIPERACILLIN/TAZOBACTAM,ERAVACYCLINE,escalation,37.300000,mixed,NaN,None
3,CEFTRIAXON,ERAVACYCLINE,escalation,37.200000,mixed,NaN,None
4,PIPERACILLIN/TAZOBACTAM,CEFOTETAN,escalation,33.320988,mixed,NaN,None
5,AZTREONAM,CEFOTETAN,escalation,32.520000,mixed,NaN,None
6,CEFTRIAXON,CEFOTETAN,escalation,31.828571,mixed,NaN,None
7,CEFTAZIDIM,CEFOTETAN,escalation,28.972973,mixed,NaN,None
8,CEFUROXIM,CEFOTETAN,escalation,28.096154,mixed,NaN,None
9,CEFTAZIDIM,FOSFOMYCIN,escalation,26.847458,mixed,NaN,None


## 8. Adjusted Downstream-Testing Regression Input Audit

This is the key corrected behavior.

The adjusted model should use only pairs with validation status `robust` or `supported`. It should not receive all support-passing edges.


In [10]:
if not escalation_results.empty and not validation_results.empty:
    validated_edge_results = CascadeAnalysisWorkflow._validated_edge_results(escalation_results, validation_results)
    valid_keys = validated_edge_results[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()
    all_support_keys = escalation_results.loc[escalation_results["passes_support_threshold"].eq(True), ["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()

    print("support-passing candidate edges:", len(all_support_keys))
    print("validated robust/supported edges entering adjusted model:", len(valid_keys))
    display(validated_edge_results.head(20))
else:
    validated_edge_results = pd.DataFrame()
    print("Cannot construct adjusted-model input because escalation or validation results are unavailable.")


support-passing candidate edges: 835
validated robust/supported edges entering adjusted model: 0


,upstream_antibiotic,downstream_antibiotic,upstream_result_group_x,positive_support_n,positive_tested_n,positive_probability,upstream_result_group_y,negative_support_n,negative_tested_n,negative_probability,total_support_n,escalation_ratio,passes_support_threshold


In [11]:
adjusted_diagnostics = read_parquet_if_exists(expected["cascade_adjusted_diagnostics"])

if not adjusted_diagnostics.empty and not validation_results.empty:
    modeled_keys = adjusted_diagnostics[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()
    expected_keys = CascadeAnalysisWorkflow._validated_edge_results(escalation_results, validation_results)[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates() if not escalation_results.empty else pd.DataFrame(columns=["upstream_antibiotic", "downstream_antibiotic"])

    extra = modeled_keys.merge(expected_keys, on=["upstream_antibiotic", "downstream_antibiotic"], how="left", indicator=True).query("_merge == 'left_only'")
    missing = expected_keys.merge(modeled_keys, on=["upstream_antibiotic", "downstream_antibiotic"], how="left", indicator=True).query("_merge == 'left_only'")

    print("adjusted diagnostics rows:", len(adjusted_diagnostics))
    print("modeled pair keys:", len(modeled_keys))
    print("unexpected adjusted pairs not robust/supported:", len(extra))
    print("validated pairs missing adjusted diagnostic row:", len(missing))
    if "non_estimable_reason" in adjusted_diagnostics.columns:
        display(adjusted_diagnostics["non_estimable_reason"].fillna("estimable").value_counts().rename_axis("reason").reset_index(name="pair_n"))
    else:
        print("ISSUE: adjusted diagnostics artifact lacks non_estimable_reason; it was generated before the current model-audit schema.")
    if len(extra):
        print("ISSUE: adjusted diagnostics contain pairs outside robust/supported validation set; regenerate cascade outputs.")
        display(extra.head(20))
else:
    print("No adjusted diagnostics artifact yet, or validation missing. This is expected before the next cascade run.")


adjusted diagnostics rows: 558
modeled pair keys: 558
unexpected adjusted pairs not robust/supported: 558
validated pairs missing adjusted diagnostic row: 0
ISSUE: adjusted diagnostics artifact lacks non_estimable_reason; it was generated before the current model-audit schema.
ISSUE: adjusted diagnostics contain pairs outside robust/supported validation set; regenerate cascade outputs.


,upstream_antibiotic,downstream_antibiotic,_merge
0,CEFEPIM,ERAVACYCLINE,left_only
1,CEFOTAXIM,CEFOTETAN,left_only
2,GENTAMICIN,FOSFOMYCIN,left_only
3,MINOCYCLIN,DORIPENEM,left_only
4,CEFOTAXIM,MEROPENEM/VABORBACTAM,left_only
5,CEFEPIM,MEROPENEM/VABORBACTAM,left_only
6,GENTAMICIN,CEFOTETAN,left_only
7,MINOCYCLIN,AZTREONAM,left_only
8,NITROFURANTOIN,TIGECYCLIN,left_only
9,NITROFURANTOIN,CEFTOLOZANE/TAZOBACTAM,left_only


## 9. Covariates Entering the Adjusted Model

The primary adjusted model uses leakage-safe episode covariates. Lab/vital summaries are excluded from primary adjustment because the extracts do not prove they occurred before culture order.

Comorbidity is used in two forms:

- `cov_comorbidity_count`: active comorbidity burden at culture.
- `comorb_*`: frequent active component flags, included only when supported inside a given pair model.


In [12]:
from amr_cascade_platform.cascade.statistics.cascade_covariate_builder import CascadeCovariateBuilder

if not culture_episodes.empty:
    covariates = CascadeCovariateBuilder(settings, path_manager).build(culture_episodes)
    print("covariate rows:", len(covariates), "columns:", len(covariates.columns))
    comorb_cols = sorted([c for c in covariates.columns if c.startswith("comorb_")])
    timing_limited = sorted([c for c in covariates.columns if c in DownstreamTestingRegression._TIMING_LIMITED_ACUITY_COVARIATES])
    print("comorb_* component columns:", len(comorb_cols))
    print("timing-limited lab/vital columns present but excluded from primary model:", len(timing_limited))
    display(pd.DataFrame({"comorbidity_component_columns": comorb_cols[:50]}))
    display(covariates.head())
else:
    covariates = pd.DataFrame()
    print("culture_episodes missing; cannot build covariates.")


covariate rows: 74889 columns: 57
comorb_* component columns: 20
timing-limited lab/vital columns present but excluded from primary model: 12


,comorbidity_component_columns
0,comorb_abdominal_pain_and_other_digestive_abdo...
1,comorb_abnormal_findings_without_diagnosis
2,comorb_cardiac_dysrhythmias
3,comorb_chronic_kidney_disease
4,comorb_congestive_heart_failure
5,comorb_cystic_fibrosis
6,comorb_diabetes_complicated
7,comorb_diabetes_mellitus_without_complication
8,comorb_diabetes_uncomplicated
9,comorb_disorders_of_lipid_metabolism


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered,organism,source_site,cov_calendar_year,cov_calendar_month,cov_age_bin,cov_sex,...,comorb_implant_device_or_graft_related_encounter,comorb_medical_examination_evaluation,comorb_musculoskeletal_pain_not_low_back_pain,comorb_organ_transplant_status,comorb_other_general_signs_and_symptoms,comorb_other_specified_status,comorb_personal_family_history_of_disease,comorb_renal_failure,comorb_respiratory_signs_and_symptoms,comorb_solid_tumor_without_metastasis
0,JC1853501,131251681302,564085701,2018-06-02 22:57:00+00:00,ESCHERICHIA COLI,armd,2018,6,unknown,unknown,...,0,0,0,0,0,0,0,0,0,0
1,JC739701,131247304008,552834814,2018-03-03 06:12:00+00:00,ESCHERICHIA COLI,armd,2018,3,unknown,unknown,...,0,0,0,0,0,0,0,0,0,0
2,JC650342,131024511713,419486271,2013-05-25 06:39:00+00:00,ESCHERICHIA COLI,armd,2013,5,unknown,unknown,...,0,0,0,0,0,0,0,0,0,0
3,JC1576587,131337242219,809023357,2022-07-29 16:48:00+00:00,ESCHERICHIA COLI,armd,2022,7,unknown,unknown,...,0,0,0,0,0,1,0,0,0,0
4,JC1816043,131242643128,545840578,2018-01-05 05:08:00+00:00,ESCHERICHIA COLI,armd,2018,1,unknown,unknown,...,0,0,0,0,0,0,0,0,0,0


In [13]:
# Inspect which covariates would enter one adjusted pair model.
if not filtered_pairs.empty and not validated_edge_results.empty and not culture_episodes.empty:
    regression = DownstreamTestingRegression(settings, path_manager)
    first = validated_edge_results.iloc[0]
    pair_frame = filtered_pairs[
        (filtered_pairs["upstream_antibiotic"].eq(first["upstream_antibiotic"]))
        & (filtered_pairs["downstream_antibiotic"].eq(first["downstream_antibiotic"]))
    ].copy()
    pair_frame["upstream_positive"] = pair_frame["upstream_susceptibility"].map(regression._map_positive)
    pair_frame = pair_frame[pair_frame["upstream_positive"].isin([0, 1])]
    pair_frame = pair_frame.merge(covariates, on=join_keys, how="left", validate="many_to_one")
    covs_for_pair = regression._adjustment_covariates_for_frame(pair_frame)
    design = regression._build_design_matrix(pair_frame, include_upstream_positive=True)
    print("example pair:", first["upstream_antibiotic"], "->", first["downstream_antibiotic"])
    print("model rows:", len(pair_frame))
    print("covariates selected before dummy expansion:", len(covs_for_pair))
    print(covs_for_pair)
    print("design matrix shape:", None if design is None else design.shape)
    if design is not None:
        display(design.head())
else:
    print("Need filtered pairs, validated edges, and culture episodes to inspect pair-specific design matrix.")


Need filtered pairs, validated edges, and culture episodes to inspect pair-specific design matrix.


## 10. Prevalence-Shift Input Audit

Correct production behavior:

- Use only downstream antibiotics from validated `robust` / `supported` **escalation** edges.
- Do not use upstream-only drugs, unrelated eligible drugs, or validated suppression edges as primary prevalence targets.


In [14]:
from amr_cascade_platform.surveillance.prevalence_shift_analyzer import PrevalenceShiftAnalyzer

prevalence_analyzer = PrevalenceShiftAnalyzer(settings)
validated_downstream = prevalence_analyzer._validated_downstream_antibiotics(edge_report if not edge_report.empty else validation_results)
print("validated downstream escalation antibiotics for prevalence:", len(validated_downstream))
print(sorted(validated_downstream))

prevalence_shift = read_parquet_if_exists(expected["prevalence_shift"])
mnar_curve = read_parquet_if_exists(expected["prevalence_mnar_curve"])
delta_curve = read_parquet_if_exists(expected["prevalence_legacy_delta_curve"])

if not prevalence_shift.empty:
    reported_drugs = set(prevalence_shift["drug"].dropna().astype(str))
    extra_drugs = sorted(reported_drugs - set(validated_downstream))
    missing_drugs = sorted(set(validated_downstream) - reported_drugs)
    print("prevalence result drugs:", len(reported_drugs))
    print("drugs reported but not validated downstream escalation drugs:", extra_drugs)
    print("validated downstream drugs absent from prevalence result, usually due support thresholds:", missing_drugs)
    display(prevalence_shift.head(20))
else:
    print("No prevalence_shift.parquet yet. This is expected before rerunning prevalence after the code update.")


validated downstream escalation antibiotics for prevalence: 0
[]
prevalence result drugs: 22
drugs reported but not validated downstream escalation drugs: ['AMOXICILLIN/CLAVULANIC ACID', 'AMPICILLIN', 'AMPICILLIN/SULBACTAM', 'AZTREONAM', 'CEFALOTIN', 'CEFAZOLIN', 'CEFEPIM', 'CEFOTAXIM', 'CEFOXITIN', 'CEFTAZIDIM', 'CEFTRIAXON', 'CEFUROXIM', 'CIPROFLOXACIN', 'CO-TRIMOXAZOL', 'GENTAMICIN', 'LEVOFLOXACIN', 'MINOCYCLIN', 'MOXIFLOXACIN', 'NITROFURANTOIN', 'PIPERACILLIN/TAZOBACTAM', 'TETRACYCLIN', 'TOBRAMYCIN']
validated downstream drugs absent from prevalence result, usually due support thresholds: []


,organism,drug,eligible_n,observed_n,tested_n,evaluable_tested_n,unobserved_n,untested_n,non_evaluable_observed_n,unknown_binary_outcome_n,...,mean_abs_smd_independent_vs_untested,mean_abs_smd_cascade_vs_untested,mnar_lambda0_prevalence,mnar_lambda0_prevalence_pct,mnar_lambda0_shift_from_naive,mnar_lambda0_shift_from_naive_pct,mnar_lambda0_absolute_shift_pct,mnar_status_lambda0,mnar_feature_count,mnar_feature_columns
0,ESCHERICHIA COLI,CEFUROXIM,74889,2933,2720,2720,71956,71956,213,72169,...,0.109952,NaN,0.534130,53.412989,-0.447365,-44.736519,44.736519,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
1,ESCHERICHIA COLI,CEFTAZIDIM,74889,2933,2905,2905,71956,71956,28,71984,...,0.063142,NaN,0.534625,53.462537,-0.443748,-44.374757,44.374757,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
2,ESCHERICHIA COLI,MOXIFLOXACIN,55363,552,549,549,54811,54811,3,54814,...,0.129099,NaN,0.603189,60.318854,-0.441076,-44.107560,44.107560,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
3,ESCHERICHIA COLI,AZTREONAM,74889,2613,2595,2595,72276,72276,18,72294,...,0.068306,NaN,0.477499,47.749941,-0.412760,-41.275953,41.275953,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
4,ESCHERICHIA COLI,TOBRAMYCIN,74889,4158,3846,3846,70731,70731,312,71043,...,0.029815,NaN,0.477604,47.760398,-0.398821,-39.882082,39.882082,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
5,ESCHERICHIA COLI,MINOCYCLIN,29242,584,565,565,28658,28658,19,28677,...,0.013478,NaN,0.455189,45.518897,-0.395012,-39.501198,39.501198,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
6,ESCHERICHIA COLI,NITROFURANTOIN,74889,5858,5739,5739,69031,69031,119,69150,...,0.015753,NaN,0.401523,40.152286,-0.383401,-38.340123,38.340123,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
7,ESCHERICHIA COLI,AMOXICILLIN/CLAVULANIC ACID,74889,4555,4088,4088,70334,70334,467,70801,...,0.019204,NaN,0.455903,45.590269,-0.383006,-38.300641,38.300641,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
8,ESCHERICHIA COLI,CEFEPIM,74889,2966,2963,2963,71923,71923,3,71926,...,0.042865,NaN,0.441942,44.194151,-0.379505,-37.950479,37.950479,estimated,43,"organism, source_site, cov_calendar_year, cov_..."
9,ESCHERICHIA COLI,PIPERACILLIN/TAZOBACTAM,74889,5970,5864,5864,68919,68919,106,69025,...,0.014339,NaN,0.384808,38.480760,-0.370995,-37.099450,37.099450,estimated,43,"organism, source_site, cov_calendar_year, cov_..."


## 11. Interpreting Sensitivity Output Correctly

Do not require every sensitivity value to be below naive prevalence. That would be wrong.

Expected behavior:

- Lower bound is always `<= naive` because unknown binary outcomes are treated as susceptible.
- Upper bound is usually `>= naive` because unknown binary outcomes are treated as resistant.
- In MNAR odds-tilt sensitivity, positive `lambda` lowers expected resistance among unknown outcomes; negative `lambda` raises it.
- Therefore the positive-selection side should often be below naive, but the full curve intentionally spans assumptions above and below naive.


In [15]:
if not prevalence_shift.empty:
    check = prevalence_shift.copy()
    check["lower_le_naive"] = check["prevalence_lower_bound"].le(check["naive_prevalence"])
    check["upper_ge_lower"] = check["prevalence_upper_bound"].ge(check["prevalence_lower_bound"])
    display(check[["organism", "drug", "naive_prevalence_pct", "prevalence_lower_bound_pct", "prevalence_upper_bound_pct", "mnar_lambda0_prevalence_pct", "standardised_prevalence_pct", "lower_le_naive", "upper_ge_lower"]].head(30))
    print("all lower bounds <= naive:", bool(check["lower_le_naive"].all()))
    print("all upper bounds >= lower bounds:", bool(check["upper_ge_lower"].all()))

if not mnar_curve.empty:
    display(mnar_curve.groupby("mnar_lambda").agg(
        drugs=("drug", "nunique"),
        mean_shift_from_naive_pct=("mnar_shift_from_naive_pct", "mean"),
        median_shift_from_naive_pct=("mnar_shift_from_naive_pct", "median"),
        estimated_rows=("mnar_status", lambda s: int((s == "estimated").sum())),
    ).reset_index())


,organism,drug,naive_prevalence_pct,prevalence_lower_bound_pct,prevalence_upper_bound_pct,mnar_lambda0_prevalence_pct,standardised_prevalence_pct,lower_le_naive,upper_ge_lower
0,ESCHERICHIA COLI,CEFUROXIM,8.676471,0.315133,96.683091,53.412989,8.676471,True,True
1,ESCHERICHIA COLI,CEFTAZIDIM,9.087780,0.352522,96.473447,53.462537,9.087780,True,True
2,ESCHERICHIA COLI,MOXIFLOXACIN,16.211293,0.160757,99.169120,60.318854,16.211293,True,True
3,ESCHERICHIA COLI,AZTREONAM,6.473988,0.224332,96.759204,47.749941,6.473988,True,True
4,ESCHERICHIA COLI,TOBRAMYCIN,7.878315,0.404599,95.268998,47.760398,7.878315,True,True
5,ESCHERICHIA COLI,MINOCYCLIN,6.017699,0.116271,98.184119,45.518897,6.017699,True,True
6,ESCHERICHIA COLI,NITROFURANTOIN,1.812162,0.138872,92.475530,40.152286,1.812162,True,True
7,ESCHERICHIA COLI,AMOXICILLIN/CLAVULANIC ACID,7.289628,0.397922,94.939177,45.590269,7.289628,True,True
8,ESCHERICHIA COLI,CEFEPIM,6.243672,0.247032,96.290510,44.194151,6.243672,True,True
9,ESCHERICHIA COLI,PIPERACILLIN/TAZOBACTAM,1.381310,0.108160,92.277905,38.480760,1.381310,True,True


all lower bounds <= naive: True
all upper bounds >= lower bounds: True


,mnar_lambda,drugs,mean_shift_from_naive_pct,median_shift_from_naive_pct,estimated_rows
0,-2.0,22,-65.814230,-71.889814,22
1,-1.0,22,-50.916345,-56.216562,22
2,-0.5,22,-41.013427,-46.517583,22
3,0.0,22,-30.371426,-36.247597,22
4,0.5,22,-20.020048,-25.591752,22
5,1.0,22,-10.858964,-17.181382,22
6,2.0,22,2.330452,-4.046033,22


## 12. Optional: Run Small Local Stages

This block is deliberately disabled. Only set `RUN_OPTIONAL_STAGES = True` when using a small test dataset or when you explicitly want the notebook to regenerate local artifacts.


In [16]:
if RUN_OPTIONAL_STAGES:
    from amr_cascade_platform.data.gold.gold_build_manager import GoldBuildManager, GoldBuildRequest
    from amr_cascade_platform.data.pipelines.gold_pipeline import GoldPipeline
    from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisRequest
    from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisWorkflow
    from amr_cascade_platform.surveillance.workflows.prevalence_shift_workflow import PrevalenceShiftWorkflow, PrevalenceShiftRequest

    GoldPipeline(GoldBuildManager(settings, path_manager)).run(
        GoldBuildRequest(source_scope="combined", site=None, organism=ORGANISM_LABEL)
    )
    CascadeAnalysisWorkflow(settings, path_manager).run(
        CascadeAnalysisRequest(gold_scope="combined", site=None, organism=ORGANISM_SLUG)
    )
    PrevalenceShiftWorkflow(settings, path_manager).run(
        PrevalenceShiftRequest(scope="combined", site=None, organism=ORGANISM_SLUG)
    )
else:
    print("Optional local stage execution disabled. Set RUN_OPTIONAL_STAGES = True to run it.")


Optional local stage execution disabled. Set RUN_OPTIONAL_STAGES = True to run it.


## 13. HPC Readiness Checklist

Before production HPC:

1. No old jobs should be running from the previous code contract.
2. `.venv` on HPC should be recreated cleanly, not reused from the broken Python 3.9/3.12 mixed environment.
3. Gold, cascade, validation merge, adjusted diagnostics, prevalence, and reports must be regenerated.
4. After production, rerun this notebook against the HPC-returned artifacts or copied `logs_outputs` to verify:
   - adjusted diagnostics pair keys are a subset of robust/supported validation keys;
   - prevalence drugs are a subset of validated downstream escalation drugs;
   - lower bounds are <= naive;
   - full MNAR/delta curves are interpreted as sensitivity ranges, not forced corrections.
